In [4]:
# 1. Imports and Setup
import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


/var/folders/q7/3g65n_191jxgh71z8qlfr9th00v5bg/T/ipykernel_73064/3369112428.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
/Users/sauravprateek/Documents/saurav-codes/neural-nets-codelab/saurav-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
google_open_internship_roles = [
    'https://www.google.com/about/careers/applications/jobs/results/120997883141857990-software-engineering-intern-summer-2027'
]

content = []

for role in google_open_internship_roles:
    loader = WebBaseLoader("https://www.google.com/about/careers/applications/jobs/results/120997883141857990-software-engineering-intern-summer-2027")
    docs = loader.load()
    content.extend(docs)

In [7]:
print(content[0].page_content)

Software Engineering Intern, Summer 2027 — Google CareersCareersCareersSkip navigation linkshomehomeHomeHomework_outlinework_outlineJobsJobsnoogler_hatnoogler_hatStudentsStudentsgooglegoogleHow we workHow we workhandymanhandymanHow we hireHow we hireperson_outlineperson_outlineYour careerYour careerhelp_outlineHelp linkfeedbackSend feedbackmore_vert HelpSend FeedbackSign inCareersCareershomeHomework_outlineJobsexpand_morenoogler_hatStudentsexpand_moregoogleHow we workexpand_morehandymanHow we hireexpand_moreperson_outlineYour careerexpand_morejob detailsarrow_backBack to jobs searchJobs search results3,770  jobs matchedStaff Software Engineer, Vector Search, Vertex AIWarsaw, PolandYouTube Ads Action Demand Verticals Tech LeadMountain View, CA, USAForward Deployed Engineer IV, GenAI, Google CloudSan Francisco, CA, USA; Atlanta, GA, USA; +24 more; +23 moreStaff Developer Experience Engineer, DeepMindBengaluru, Karnataka, IndiaPrincipal Growth Manager, AppDev, Google Customer SolutionsNew

In [8]:
len(content)

1

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50
)
chunks = text_splitter.split_documents(content)

In [10]:
len(chunks)

27

In [11]:
chunks[10].page_content

"Currently pursuing a Bachelor's, Master's, or Dual degree program in Computer Science, or a related technical field.\nExperience in one or more of the following: architecting or developing distributed systems, concurrency, multi-threading, or synchronization.\nExperience with one or more general purpose programming languages (e.g., Java, C/C++, Python, JavaScript, Go, etc.).\nExperience with data structures, algorithms, and software design.\nPreferred qualifications:"

In [13]:
import uuid
import chromadb

# Embed and store in local Chroma DB
embedding_model_name = 'gemini-embedding-2'
embeddings_model = GoogleGenerativeAIEmbeddings(model='gemini-embedding-2')

embeddings = [embeddings_model.embed_query(chunk.page_content) for chunk in chunks]

text_embedding_pairs = list(zip(chunks, embeddings))
chroma_client = chromadb.EphemeralClient()
collection = chroma_client.create_collection(name="gemini_fixed_collection")

collection.add(
    documents=[chunk.page_content for chunk in chunks],
    embeddings=embeddings,
    metadatas=[chunk.metadata for chunk in chunks],
    ids=[str(uuid.uuid4()) for _ in chunks]
)

vectorstore = Chroma(
    client=chroma_client,
    collection_name="gemini_fixed_collection",
    embedding_function=embeddings_model
)

In [15]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [19]:
# 3. The RAG Chain Setup
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Combine components into a single chain
qa_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, qa_chain)

# 4. Execution / Runtime
response = rag_chain.invoke({
    "input": "What is the minimum qualification for the job role?"
})

print(response["answer"])

Based on the provided context, the minimum qualifications for the job role are:

* **Education:** Currently pursuing a Bachelor's, Master's, or Dual degree program in Computer Science or a related technical field.
* **Technical Experience:** 
  * Experience with one or more general-purpose programming languages (e.g., Java, C/C++, Python, JavaScript, Go, etc.).
  * Experience with data structures, algorithms, and software design.
  * Experience in one or more of the following: architecting or developing distributed systems, concurrency, multi-threading, or synchronization.


In [21]:
related_documents = retriever.invoke('What is the minimum qualification for the job role?')

In [22]:
len(related_documents)

3

In [29]:
related_documents[1].page_content

'Preferred qualifications:\nCurrently in your penultimate year of study.\nExperience in architecture, artificial intelligence, compilers, database, data mining, distributed systems, machine learning, networking, or systems.\nExperience in designing and implementing a complex system, for production or experimental use.\nExperience with performance, reliability, systems data analysis, visualization tools, or debugging.\nExcellent engineering skills.'